In [7]:
import pandas as pd
import numpy as np
# Datei einlesen
file_path = "01_Realisierte_Erzeugung_201901010000_202601010000_Monat.csv"
df = pd.read_csv(file_path, sep=";", encoding="latin1", on_bad_lines="skip")
display(df)

,ï»¿Datum von,Datum bis,Biomasse [MWh] Berechnete AuflÃ¶sungen,Wasserkraft [MWh] Berechnete AuflÃ¶sungen,Wind Offshore [MWh] Berechnete AuflÃ¶sungen,Wind Onshore [MWh] Berechnete AuflÃ¶sungen,Photovoltaik [MWh] Berechnete AuflÃ¶sungen,Sonstige Erneuerbare [MWh] Berechnete AuflÃ¶sungen,Kernenergie [MWh] Berechnete AuflÃ¶sungen,Braunkohle [MWh] Berechnete AuflÃ¶sungen,Steinkohle [MWh] Berechnete AuflÃ¶sungen,Erdgas [MWh] Berechnete AuflÃ¶sungen,Pumpspeicher [MWh] Berechnete AuflÃ¶sungen,Sonstige Konventionelle [MWh] Berechnete AuflÃ¶sungen
0,01.01.2019,01.02.2019,"3.550.046,75","1.200.910,75","2.213.086,50","12.522.275,25","728.156,75","136.702,50","6.821.064,00","9.862.384,25","7.549.348,25","5.740.661,50","824.704,25","1.222.620,50"
1,01.02.2019,01.03.2019,"3.215.899,50","1.032.343,75","1.938.096,25","8.892.392,75","2.184.288,25","137.206,75","6.196.593,00","9.835.908,50","5.566.684,00","4.474.908,25","655.388,50","1.207.127,00"
2,01.03.2019,01.04.2019,"3.481.646,50","1.384.887,50","2.540.870,25","13.696.172,50","3.020.968,75","145.525,50","6.713.072,25","8.298.238,75","3.119.376,25","3.647.710,75","788.373,00","1.159.437,75"
3,01.04.2019,01.05.2019,"3.324.551,25","1.359.297,25","1.707.845,50","7.284.558,75","5.015.643,00","132.009,25","5.692.459,75","9.075.669,75","3.411.070,00","3.643.917,25","753.675,00","1.029.269,25"
4,01.05.2019,01.06.2019,"3.393.611,75","1.543.969,00","1.789.633,75","6.411.664,00","5.019.334,00","120.702,25","4.610.711,25","8.654.967,00","3.494.879,50","3.367.003,50","650.016,25","1.089.072,25"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
79,01.08.2025,01.09.2025,"2.852.499,35","1.463.011,45","1.699.077,95","5.459.159,51","9.672.278,48","73.973,16",-,"4.518.747,75","1.301.445,00","3.015.974,75","843.045,25","1.075.939,50"
80,01.09.2025,01.10.2025,"2.794.210,42","1.208.876,63","2.410.541,28","9.721.142,41","6.435.895,39","72.157,99",-,"4.925.688,75","1.397.569,00","3.790.773,75","733.651,25","1.014.139,00"
81,01.10.2025,01.11.2025,"3.034.056,45","1.116.769,65","2.901.834,16","13.695.534,40","3.642.622,45","79.788,30",-,"5.648.721,13","2.166.156,53","5.275.413,52","756.737,01","1.394.098,94"
82,01.11.2025,01.12.2025,"3.069.622,61","1.084.135,10","2.716.300,26","9.952.192,85","2.210.900,10","81.528,93",-,"6.595.504,11","3.165.924,61","7.519.788,06","673.562,25","1.350.662,80"


In [8]:
import pandas as pd

# ---------- LOAD ----------
df = pd.read_csv(
    file_path,
    sep=";",
    encoding="utf-8-sig",
    decimal=",",
    thousands=".",
    na_values=["-"]
)

# ---------- RENAME COLUMNS ----------
df.columns = (
    df.columns
    .str.replace(r"\[.*?\]", "", regex=True)   # remove [MWh]
    .str.replace("Berechnete Auflösungen", "", regex=False)
    .str.strip()
    .str.replace(" ", "_")
    .str.lower()
)

# ---------- FIX DATE ----------
def fix_date(x):
    x = str(int(x)).zfill(8)  # 1012019 → 01012019
    return pd.to_datetime(x, format="%d%m%Y")

df["date_from"] = df["datum_von"].apply(fix_date)
df["date_to"] = df["datum_bis"].apply(fix_date)

df = df.drop(columns=["datum_von", "datum_bis"])

# ---------- MELT (WIDE → LONG) ----------
df_long = df.melt(
    id_vars=["date_from", "date_to"],
    var_name="energy_source",
    value_name="generation_mwh"
)

# ---------- CLEAN ENERGY NAMES ----------
df_long["energy_source"] = df_long["energy_source"].str.replace("_", " ")

# ---------- SORT ----------
df_long = df_long.sort_values(["date_from", "energy_source"])

df_long.head()

,date_from,date_to,energy_source,generation_mwh
0,2019-01-01,2019-02-01,biomasse,3550046.75
588,2019-01-01,2019-02-01,braunkohle,9862384.25
756,2019-01-01,2019-02-01,erdgas,5740661.50
504,2019-01-01,2019-02-01,kernenergie,6821064.00
336,2019-01-01,2019-02-01,photovoltaik,728156.75


In [9]:
df_long["year"] = df_long["date_from"].dt.year
df_long["month"] = df_long["date_from"].dt.month

In [10]:
df_long["year"] = df_long["date_from"].dt.year
df_long["month"] = df_long["date_from"].dt.month

# Replace missing nuclear generation values with 0
# Nach dem Atomausstieg bedeutet NaN bei Kernenergie nicht "fehlende Daten",
# sondern tatsächliche Stromerzeugung = 0 MWh

df_long.loc[
    (df_long["energy_source"] == "kernenergie") & (df_long["generation_mwh"].isna()),
    "generation_mwh"
] = 0

In [11]:
display(df_long)

,date_from,date_to,energy_source,generation_mwh,year,month
0,2019-01-01,2019-02-01,biomasse,3550046.75,2019,1
588,2019-01-01,2019-02-01,braunkohle,9862384.25,2019,1
756,2019-01-01,2019-02-01,erdgas,5740661.50,2019,1
504,2019-01-01,2019-02-01,kernenergie,6821064.00,2019,1
336,2019-01-01,2019-02-01,photovoltaik,728156.75,2019,1
...,...,...,...,...,...,...
1007,2025-12-01,2026-01-01,sonstige konventionelle,1429378.01,2025,12
755,2025-12-01,2026-01-01,steinkohle,2599886.52,2025,12
167,2025-12-01,2026-01-01,wasserkraft,1022205.69,2025,12
251,2025-12-01,2026-01-01,wind offshore,3425383.69,2025,12


In [12]:
df_long[
    (df_long["energy_source"] == "kernenergie") &
    (df_long["year"] >= 2023)
].tail()

,date_from,date_to,energy_source,generation_mwh,year,month
583,2025-08-01,2025-09-01,kernenergie,0.0,2025,8
584,2025-09-01,2025-10-01,kernenergie,0.0,2025,9
585,2025-10-01,2025-11-01,kernenergie,0.0,2025,10
586,2025-11-01,2025-12-01,kernenergie,0.0,2025,11
587,2025-12-01,2026-01-01,kernenergie,0.0,2025,12


In [13]:
df_long = df_long.rename(columns={
    "date_from": "datum_von",
    "date_to": "datum_bis",
    "energy_source": "energietraeger",
    "generation_mwh": "erzeugung_mwh",
    "year": "jahr",
    "month": "monat"
})

In [14]:
df_long["energietraeger"] = df_long["energietraeger"].replace({
    "biomasse": "Biomasse",
    "wasserkraft": "Wasserkraft",
    "wind offshore": "Wind offshore",
    "wind onshore": "Wind onshore",
    "photovoltaik": "Photovoltaik",
    "sonstige erneuerbare": "Sonstige Erneuerbare",
    "kernenergie": "Kernenergie",
    "braunkohle": "Braunkohle",
    "steinkohle": "Steinkohle",
    "erdgas": "Erdgas",
    "pumpspeicher": "Pumpspeicher",
    "sonstige konventionelle": "Sonstige Konventionelle"
})

In [15]:
display(df_long.head())
display(df_long.tail())
df_long.info()
df_long["energietraeger"].unique()

,datum_von,datum_bis,energietraeger,erzeugung_mwh,jahr,monat
0,2019-01-01,2019-02-01,Biomasse,3550046.75,2019,1
588,2019-01-01,2019-02-01,Braunkohle,9862384.25,2019,1
756,2019-01-01,2019-02-01,Erdgas,5740661.50,2019,1
504,2019-01-01,2019-02-01,Kernenergie,6821064.00,2019,1
336,2019-01-01,2019-02-01,Photovoltaik,728156.75,2019,1


,datum_von,datum_bis,energietraeger,erzeugung_mwh,jahr,monat
1007,2025-12-01,2026-01-01,Sonstige Konventionelle,1429378.01,2025,12
755,2025-12-01,2026-01-01,Steinkohle,2599886.52,2025,12
167,2025-12-01,2026-01-01,Wasserkraft,1022205.69,2025,12
251,2025-12-01,2026-01-01,Wind offshore,3425383.69,2025,12
335,2025-12-01,2026-01-01,Wind onshore,12474033.51,2025,12


<class 'pandas.core.frame.DataFrame'>
Index: 1008 entries, 0 to 335
Data columns (total 6 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   datum_von       1008 non-null   datetime64[ns]
 1   datum_bis       1008 non-null   datetime64[ns]
 2   energietraeger  1008 non-null   object        
 3   erzeugung_mwh   1008 non-null   float64       
 4   jahr            1008 non-null   int32         
 5   monat           1008 non-null   int32         
dtypes: datetime64[ns](2), float64(1), int32(2), object(1)
memory usage: 47.2+ KB


array(['Biomasse', 'Braunkohle', 'Erdgas', 'Kernenergie', 'Photovoltaik',
       'Pumpspeicher', 'Sonstige Erneuerbare', 'Sonstige Konventionelle',
       'Steinkohle', 'Wasserkraft', 'Wind offshore', 'Wind onshore'],
      dtype=object)

In [16]:
df_long.to_csv(
    "01_bereinigte_erzeugung_LV.csv",
    index=False,
    encoding="utf-8-sig"
)